<a href="https://colab.research.google.com/github/mwanyambu/translator/blob/main/translator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Force stable dependencies first to prevent the Choice error path
!pip install "typer>=0.9.0,<0.10.0" click==8.1.7
!pip install transformers torch sentencepiece gradio openpyxl gTTS

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.9/97.9 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 4.1 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.1.8
    Uninstalling click-8.1.8:
      Successfully uninstalled click-8.1.8
  Attempting uninstall: typer
    Found existing installation: typer 0.25.1
    Uninstalling typer-0.25.1:
      Successfully uninstalled typer-0.25.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires typer<1.0,>=0.12, but you have typer 0.9.4 which is incompatible.
huggingface-hub 1.20.1 requires click>=8.4.0, but you have click 8.1.7 which is incompatible.
huggingface-hub 1.20.1 requires typer<0.26.0,>=0.20.0, but you have typer 0.9.4 which is incompatible.
google-adk 2.3.0 requires click<9,>=8.1.8, but you have click 8.1.7 which is incompa

INFO: pip is looking at multiple versions of huggingface-hub to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of huggingface-hub to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.6/122.6 kB 10.7 MB/s eta 0:00:00
Traceback (most recent call last):
  File "/usr/lib/python3.12/pathlib.py", line 441, in __str__
^C


In [5]:

import re
import os
import pandas as pd
import gradio as gr
import torch
import numpy as np
from scipy.io.wavfile import write
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, VitsModel, VitsTokenizer

# 2. Setup GPU device acceleration targeting
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f" Audio translation pipeline active on: {device.upper()}")

# 3. Load and Align Your Uploaded Excel Files in Memory
KNOWLEDGE_BASE = {}

def load_and_align_excel(file_name, lang_column_name):
    if os.path.exists(file_name):
        try:
            print(f" Loading uploaded file: {file_name}...")
            df = pd.read_excel(file_name).fillna("")
            df.columns = [str(c).strip() for c in df.columns]
            text_col_name = df.columns[-1]

            for _, row in df.iterrows():
                book = str(row.get('Book', '')).strip().upper()
                chap = str(row.get('Chapter', '')).strip()
                verse = str(row.get('Verse', '')).strip()
                text = str(row.get(text_col_name, '')).strip()

                if book and chap and verse and text:
                    key = f"{book}_{chap}_{verse}"
                    if key not in KNOWLEDGE_BASE:
                        KNOWLEDGE_BASE[key] = {}
                    KNOWLEDGE_BASE[key][lang_column_name] = text
            print(f"    Successfully indexed keys from {file_name}")
        except Exception as e:
            print(f"    Error reading {file_name}: {e}")
    else:
        print(f"    Warning: File '{file_name}' not found. Please upload it to Colab.")

# Align sheets natively in the cloud workspace
load_and_align_excel("swa.xlsx", "Swahili")
load_and_align_excel("luo.xlsx", "Luo")
load_and_align_excel("kik.xlsx", "Kikuyu")
load_and_align_excel("guz.xlsx", "Gusii")

print(f"\n Total parallel structural verses in database memory: {len(KNOWLEDGE_BASE)}")

# 4. Load the Core Hugging Face NLLB-200 Text Model
print("\n Loading Meta NLLB-200 AI Text Model weights onto cloud GPU...")
model_name = "facebook/nllb-200-distilled-600M"
text_tokenizer = AutoTokenizer.from_pretrained(model_name)
text_model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

# --- RE-ENGINEERED HIGH-ACCURACY VOICE MODEL ROUTING ---
LANGUAGES = {
    "English": {"code": "eng_Latn", "col": "English", "tts_model": "facebook/mms-tts-eng"},
    "Swahili": {"code": "swh_Latn", "col": "Swahili", "tts_model": "facebook/mms-tts-swh"},
    "Gĩkũyũ (Kikuyu)": {"code": "kik_Latn", "col": "Kikuyu", "tts_model": "facebook/mms-tts-kik"},
    "Dholuo (Luo)": {"code": "luo_Latn", "col": "Luo", "tts_model": "facebook/mms-tts-swh"},
    "Kĩkamba (Kamba)": {"code": "kam_Latn", "col": "Kamba", "tts_model": "facebook/mms-tts-swh"},
    "Ekegusii (Gusii)": {"code": "gaz_Latn", "col": "Gusii", "tts_model": "facebook/mms-tts-swh"},
    "Af-Soomaali (Somali)": {"code": "som_Latn", "col": "Somali", "tts_model": "facebook/mms-tts-som"},
    "Afan Oromo (Oromo)": {"code": "gaz_Latn", "col": "Oromo", "tts_model": "facebook/mms-tts-gaz"},
    "Ateso (Teso)": {"code": "teo_Latn", "col": "Teso", "tts_model": "facebook/mms-tts-swh"}
}

TTS_MODEL_CACHE = {}

def generate_native_kenyan_speech(text_to_speak, lang_key):
    """Downloads and generates speech using Meta's verified regional checkpoints."""
    tts_repo = LANGUAGES[lang_key]["tts_model"]
    output_path = "kenyan_voice_output.wav"

    try:
        # Load specific model into memory cache if not already loaded
        if tts_repo not in TTS_MODEL_CACHE:
            print(f" Downloading native regional voice synthesis checkpoint: {tts_repo}...")
            tokenizer = VitsTokenizer.from_pretrained(tts_repo)
            model = VitsModel.from_pretrained(tts_repo).to(device)
            TTS_MODEL_CACHE[tts_repo] = (tokenizer, model)
        else:
            tokenizer, model = TTS_MODEL_CACHE[tts_repo]

        # Process the localized raw characters into waveforms via the GPU
        inputs = tokenizer(text_to_speak, return_tensors="pt").to(device)
        with torch.no_grad():
            output_audio = model(**inputs).waveform

        # Extract audio matrix array back safely to CPU storage for audio encoding
        audio_numpy = output_audio.cpu().numpy().squeeze()

        # Scale to standard 16-bit PCM WAV configuration (Sample rate is fixed at 16000Hz)
        audio_int16 = (audio_numpy * 32767).astype(np.int16)
        write(output_path, 16000, audio_int16)
        return output_path
    except Exception as err:
        print(f" Native speech synthesis warning: {err}")
        return None

# 5. Hybrid Translation Engine Pipeline
def translate_engine(text, src_lang, tgt_lang):
    text_clean = text.strip()
    if not text_clean:
        return "Tafadhali andika maneno.", None

    src_col = LANGUAGES[src_lang]["col"]
    tgt_col = LANGUAGES[tgt_lang]["col"]

    final_text_output = ""
    raw_sentence_to_speak = ""

    # --- STRATEGY A: Flexible Bible Lookup ---
    found_in_kb = False
    for key, langs in KNOWLEDGE_BASE.items():
        if src_col in langs and langs[src_col]:
            bible_sentence = langs[src_col].lower()
            input_sentence = text_clean.lower()

            if bible_sentence == input_sentence or input_sentence in bible_sentence:
                if tgt_col in langs and langs[tgt_col]:
                    ref_split = key.split("_")
                    readable_ref = f"{ref_split} {ref_split}:{ref_split}"
                    raw_sentence_to_speak = langs[tgt_col]
                    final_text_output = f" [VERIFIED HUMAN BIBLE REFERENCE: {readable_ref}]\n\n{raw_sentence_to_speak}"
                    found_in_kb = True
                    break

    # --- STRATEGY B: NLLB AI Fallback ---
    if not found_in_kb:
        try:
            src_code = LANGUAGES[src_lang]["code"]
            tgt_code = LANGUAGES[tgt_lang]["code"]

            text_tokenizer.src_lang = src_code
            inputs = text_tokenizer(text_clean, return_tensors="pt").to(device)
            target_lang_id = text_tokenizer.convert_tokens_to_ids(src_code if src_code == tgt_code else tgt_code)

            with torch.no_grad():
                translated_tokens = text_model.generate(
                    **inputs,
                    forced_bos_token_id=target_lang_id,
                    max_length=512
                )

            result = text_tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)
            raw_sentence_to_speak = result[0] if isinstance(result, list) else result
            final_text_output = f" [AI GENERATED NLLB TRANSLATION]\n\n{raw_sentence_to_speak}"
        except Exception as e:
            return f" Translation Error: {str(e)}", None

    # --- META REGIONAL SPEECH EXECUTION BLOCK ---
    audio_file = generate_native_kenyan_speech(raw_sentence_to_speak, tgt_lang)

    return final_text_output, audio_file

# 6. Spin up the Public Gradio Interface
demo = gr.Interface(
    fn=translate_engine,
    inputs=[
        gr.Textbox(label="Enter word, phrase, or verse to translate:", placeholder="Andika hapa..."),
        gr.Dropdown(choices=list(LANGUAGES.keys()), label="Translate From", value="English"),
        gr.Dropdown(choices=list(LANGUAGES.keys()), label="Translate To", value="Swahili")
    ],
    outputs=[
        gr.Textbox(label="Translation Text Result"),
        gr.Audio(label=" Listen to Authentic Local Voice (Meta MMS Audio)", type="filepath")
    ],
    title=" Multilingual Kenyan Translator with Authentic Native Voices",
)

# Launching with debug=True pushes runtime logging beneath the notebook cell
demo.launch(share=True, debug=True)

 Audio translation pipeline active on: CPU
 Loading uploaded file: swa.xlsx...
    Successfully indexed keys from swa.xlsx
 Loading uploaded file: luo.xlsx...
    Successfully indexed keys from luo.xlsx
 Loading uploaded file: kik.xlsx...
    Successfully indexed keys from kik.xlsx
 Loading uploaded file: guz.xlsx...
    Successfully indexed keys from guz.xlsx

 Total parallel structural verses in database memory: 79528

 Loading Meta NLLB-200 AI Text Model weights onto cloud GPU...


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://785a41c6de81bde661.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://785a41c6de81bde661.gradio.live
